In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from part1.Util.scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [34]:
load_dotenv(override=True)

True

In [55]:
MODEL = 'deepseek-chat'
deepseek = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com/v1"
);

In [46]:
openai = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_link_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "<links>\n"
    user_prompt += "\n".join(links)
    user_prompt += "\n</links>"
    return user_prompt


In [13]:
print(get_link_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

<links>
https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edward

In [49]:
def select_relevant_links_deepseek(url):
    response = deepseek.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_link_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

def select_relevant_links_openai(url):
    response = openai.chat.completions.create(
        model='o3-2025-04-16',
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_link_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [33]:
select_relevant_links_deepseek("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [71]:
select_relevant_links_deepseek("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'models showcase', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets showcase', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces showcase', 'url': 'https://huggingface.co/spaces'},
  {'type': 'chat demo', 'url': 'https://huggingface.co/chat'},
  {'type': 'company partners', 'url': 'https://huggingface.co/facebook'}]}

## Second step: make the brochure
Assemble all the details into another prompt to diff model

In [69]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links_deepseek(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [70]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
fishaudio/s2-pro
Updated
5 days ago
•
5.41k
•
497
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
9 days ago
•
67.6k
•
735
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
6 days ago
•
86.7k
•
278
Lightricks/LTX-2.3
Updated
about 19 hours ago
•
597k
•
639
HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
Updated
12 days ago
•
238k
•
470
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.29k
Wan2.2 14B Preview
🐌
1.29k
generate a video from an image with a text prompt
Running
on
Z

Brochure system prompt

In [59]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

Construct user prompt

In [60]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    return user_prompt

In [62]:
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))


You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
fishaudio/s2-pro
Updated
5 days ago
•
5.41k
•
495
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
9 days ago
•
67.6k
•
734
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
6 days ago
•
86.7k
•
273
Lightricks/LTX-2.3
Updated
about 18 hours ago
•
597k
•
639
HauhauCS/

In [65]:
def create_brochure(company_name, url):
    response = deepseek.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [66]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

## Our Mission
Hugging Face is on a mission to democratize good machine learning. We are the central collaboration platform where the global machine learning community builds the future of AI together. Our Hub serves as the home for open-source ML, enabling anyone to share, explore, discover, and experiment with cutting-edge technology.

## What We Offer
**The World's Leading AI Platform**
- **Models:** Access and collaborate on over 2 million machine learning models
- **Datasets:** Explore more than 500,000 datasets for training and research
- **Spaces:** Discover over 1 million AI applications and demos
- **Tools:** Comprehensive platform for creating, discovering, and collaborating on ML projects

## For Enterprises & Teams
We provide scalable solutions for organizations of all sizes:

**Team Plan** ($20/user/month): SSO integration, audit logs, granular access controls, analytics, and advanced compute options.

**Enterprise Plan** (starting at $50/user/month): All Team features plus highest storage/bandwidth limits, advanced security, legal/compliance support, managed billing, and personalized support.

**Pro Plan** ($9/month): Enhanced personal experience with increased storage, priority compute, and publishing capabilities.

## Our Culture & Community
We believe in building an **open and ethical AI future** together. Our platform thrives on collaboration, with community members contributing models, datasets, research papers, and educational content. We foster learning through comprehensive courses covering LLMs, robotics, computer vision, audio, diffusion models, and more.

## Who Uses Hugging Face?
Our platform serves:
- **Machine Learning Engineers & Data Scientists** building next-generation AI
- **Researchers** pushing the boundaries of what's possible
- **Students & Educators** learning and teaching AI/ML concepts
- **Enterprises** implementing AI solutions at scale
- **Open Source Contributors** sharing their work with the world

## Join Our Team
We're growing! With 182+ team members (and counting), we're looking for passionate individuals who want to democratize machine learning. Our team includes researchers, engineers, and community builders working at the heart of the AI revolution.

## Why Hugging Face?
- **Central to the AI Revolution:** We're at the forefront of open-source AI development
- **Massive Scale:** Hosting millions of models, datasets, and applications
- **Community-Driven:** Built by and for the ML community
- **Comprehensive Learning:** Extensive educational resources and courses
- **Enterprise-Ready:** Secure, scalable solutions for organizations

Join us in building the future of AI—one commit at a time.

In [67]:
def stream_brochure(company_name, url):
    stream = deepseek.chat.completions.create(
        model='deepseek-chat',
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [68]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

## Our Mission
Hugging Face is the central collaboration platform for the global machine learning community. We empower the next generation of machine learning engineers, scientists, and end users to learn, collaborate, and share their work to build an open and ethical AI future together. Positioned at the heart of the AI revolution, we provide the essential infrastructure where innovation thrives.

## The Platform
We host the world's largest open-source AI ecosystem, featuring:
* **2M+ Models** – From cutting-edge language models to diffusion and audio models
* **500K+ Datasets** – Comprehensive data resources for training and experimentation
* **1M+ Applications (Spaces)** – Deployable AI demos and tools running on our infrastructure
* **Comprehensive Documentation & Libraries** – Including Transformers, Diffusers, Datasets, and more

Our Hub serves as a Git-based platform designed for collaboration at its core, enabling users to create, discover, and experiment with machine learning technology.

## Company Culture & Community
We foster a vibrant, open-source community where knowledge sharing is fundamental. Our culture emphasizes:
* **Collaboration First** – Building technology together across borders and organizations
* **Open Innovation** – Hosting unlimited public repositories and encouraging shared development
* **Educational Commitment** – Offering extensive learning resources including courses on LLMs, robotics, computer vision, audio, diffusion models, and more
* **Ethical AI Development** – Committed to building an open and responsible AI future

## Enterprise Solutions
For organizations scaling their AI capabilities, we offer robust enterprise-grade solutions:

**Team Plan** ($20/user/month)
* SSO and SAML integration
* Audit logs and granular access controls
* Resource groups and analytics
* Advanced compute options

**Enterprise Plan** (Starting at $50/user/month)
* All Team features plus highest storage/bandwidth limits
* Advanced security and compliance processes
* Managed billing with annual commitments
* Personalized support and legal processes

**Key Enterprise Features:**
* Storage regions for data location control
* Centralized token management
* Priority support from our team
* Inference providers with usage monitoring
* 5x ZeroGPU quota for organization members

## Learning & Development
We invest heavily in education through:
* **LLM Course** – Comprehensive large language model training
* **Robotics Course** – Building robots with LeRobot
* **Specialized Courses** – Covering computer vision, audio, diffusion, 3D ML, game development, and reinforcement learning
* **Open-Source AI Cookbook** – Practical notebooks by builders for builders
* **Community Blog** – Regular technical articles, research updates, and case studies

## Careers
Join our talented team exploring the edge of technology. We're looking for passionate individuals who want to shape the future of AI. While specific openings aren't listed in the provided content, our careers page suggests ongoing opportunities to contribute to our mission.

## Why Hugging Face?
* **Scale** – Access to millions of models, datasets, and applications
* **Community** – Join the fastest-growing ML community worldwide
* **Infrastructure** – Enterprise-grade platform with robust security and compliance
* **Innovation** – At the forefront of AI research and application development
* **Openness** – Commitment to open-source principles and collaborative development

---

**Join us in building the future of AI.**  
Explore models, deploy applications, or bring Hugging Face to your organization today.